# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saif-Ullah0/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in the raw table = one page on one day (daily grain).
For my lane I aggregate to page-month level.

Unit after aggregation: one page, one month of search data.
Time window for development: month = 2026-03 (March 2026)
Final month 2026-06 is sealed as test data, not touched.

Label: I will build a proxy label by comparing a page's
impressions in the current month vs the previous month.
A page is declining if month-over-month impressions dropped.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (knowable at decision moment):
- gsc_impressions (monthly total): past observed traffic
- gsc_avg_position: average position over the month, past
- gsc_clicks (monthly total): past observed clicks
- ctr (computed): clicks/impressions, both past values
- gsc_data_available: whether GSC data exists for this page

LABEL (proxy):
- is_declining: month-over-month impressions dropped
  (current month vs previous month)
  Proxy because we cannot observe future decay directly.

CONTEXT:
- content_hash_id: page identifier, grouping only
- client_hash_id: client identifier, grouping only
- month: time partition, filtering only

EXCLUDED:
- ga4_pageviews, ga4_sessions: too many NULLs
  (most clients GSC-only, GA4 not available)
- ai_* columns: mostly NULL in this period
- report_date: daily grain, not needed after aggregation

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
%pip install -q duckdb huggingface_hub
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token=token)

con = duckdb.connect()
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
)
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
TABLE = f"{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/data_0.parquet"

In [4]:
from huggingface_hub import list_repo_files

files = list(list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset"
))
print("Files in warehouse:")
for f in files[:20]:
    print(f)

Files in warehouse:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_da

In [6]:
sample = con.execute(f"""
SELECT * FROM read_parquet('{TABLE}')
LIMIT 3
""").df()

print("Columns:", sample.columns.tolist())
print(sample.head(3))

Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           F

**Queries** 1 2 3

QUERY 1 Grain Check (After Aggregation)

In [7]:
grain = con.execute(f"""
WITH monthly AS (
    SELECT
        content_hash_id,
        client_hash_id,
        month,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        COUNT(report_date) as days_with_data
    FROM read_parquet('{TABLE}')
    WHERE month = '{MONTH}'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id, month
)
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_pages,
    MIN(days_with_data) as min_days,
    MAX(days_with_data) as max_days
FROM monthly
""").df()

print("Grain check after aggregation to page-month:")
print(grain)
print("One row per page per month confirmed if total_rows == unique_pages")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check after aggregation to page-month:
   total_rows  unique_pages  min_days  max_days
0      176738        176738         1        31
One row per page per month confirmed if total_rows == unique_pages


QUERY 2 Row Count and Date Span

In [8]:
span = con.execute(f"""
SELECT
    month,
    COUNT(DISTINCT content_hash_id) as unique_pages,
    COUNT(DISTINCT client_hash_id) as unique_clients,
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date,
    COUNT(*) as total_daily_rows
FROM read_parquet('{TABLE}')
WHERE month = '{MONTH}'
GROUP BY month
""").df()

print("Row count and date span for", MONTH)
print(span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row count and date span for 2026-03
     month  unique_pages  unique_clients earliest_date latest_date  \
0  2026-03        331437              55    2026-03-01  2026-03-31   

   total_daily_rows  
0           9841378  



Query 3 — Availability Using IS TRUE

In [9]:
avail = con.execute(f"""
SELECT
    COUNT(*) as total_daily_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        as rows_with_gsc,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        as rows_with_ga4,
    ROUND(
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        * 100.0 / COUNT(*), 2
    ) as gsc_availability_pct
FROM read_parquet('{TABLE}')
WHERE month = '{MONTH}'
""").df()

print("Data availability (IS TRUE filter)")
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data availability (IS TRUE filter)
   total_daily_rows  rows_with_gsc  rows_with_ga4  gsc_availability_pct
0           9841378        3611061         413966                 36.69


Corrected Feature Frame + Leakage Trap

In [10]:
# Build monthly aggregated feature frame
feature_df = con.execute(f"""
WITH current_month AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as impressions_current,
        SUM(gsc_clicks) as clicks_current,
        AVG(gsc_avg_position) as avg_position,
        COUNT(report_date) as days_active
    FROM read_parquet('{TABLE}')
    WHERE month = '{MONTH}'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
),
prev_month AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as impressions_prev
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=2026-02/data_0.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT
    c.content_hash_id,
    c.impressions_current,
    c.clicks_current,
    c.avg_position,
    c.days_active,
    COALESCE(p.impressions_prev, 0) as impressions_prev,
    CASE
        WHEN COALESCE(p.impressions_prev, 0) = 0 THEN NULL
        WHEN c.impressions_current < p.impressions_prev THEN TRUE
        ELSE FALSE
    END as is_declining,
    (c.impressions_current - COALESCE(p.impressions_prev, 0))
        as impressions_change
FROM current_month c
LEFT JOIN prev_month p ON c.content_hash_id = p.content_hash_id
LIMIT 50000
""").df()

print("Feature frame shape:", feature_df.shape)
print(feature_df.head())
print("\nLabel distribution:")
print(feature_df["is_declining"].value_counts())
print("\nMissing values:")
print(feature_df.isnull().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (50000, 8)
            content_hash_id  impressions_current  clicks_current  \
0  content_4a1ca0fa5c177e0c                 14.0             0.0   
1  content_225dc9235023be5f                488.0             1.0   
2  content_cfad137c1b04251b                438.0             0.0   
3  content_9e7c70abfbae371e               2436.0             0.0   
4  content_c0fc3b40ce00a5d7                335.0             1.0   

   avg_position  days_active  impressions_prev  is_declining  \
0      4.266667           10              18.0          True   
1     17.148172           31             123.0         False   
2     10.127319           31             150.0         False   
3      5.467633           31              17.0         False   
4      1.229881           31              51.0         False   

   impressions_change  
0                -4.0  
1               365.0  
2               288.0  
3              2419.0  
4               284.0  

Label distribution:
is_declin

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Drop rows where label is null
df_clean = feature_df.dropna(subset=["is_declining"])
y = df_clean["is_declining"].astype(int)

# THE TRAP: add impressions_change as leaky feature
X_leaky = df_clean[["impressions_current","avg_position",
                      "clicks_current","impressions_change"]].fillna(0)
tree_leaky = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_leaky.fit(X_leaky, y)
print(f"Leaky score: {tree_leaky.score(X_leaky, y):.3f} <- lying")
print("impressions_change directly encodes the label direction.")

# HONEST: remove leaky feature
X_honest = df_clean[["impressions_current","avg_position",
                       "clicks_current","days_active"]].fillna(0)
tree_honest = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_honest.fit(X_honest, y)
print(f"\nHonest score: {tree_honest.score(X_honest, y):.3f} <- real")
print("impressions_change deleted. Only honest features remain.")

df_clean = df_clean.drop(columns=["impressions_change"])
print("Final columns:", df_clean.columns.tolist())

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.